In [ ]:
import os, sys, yaml
path2add = os.path.normpath(os.path.abspath(os.path.join(os.path.curdir, os.path.pardir)))
print(path2add)

if (not (path2add in sys.path)) :
    sys.path.append(path2add)

import BeamTestHelpers as helper
from pathlib import Path
from natsort import natsorted

In [ ]:
path_to_files = "path/to/directory"
files = natsorted(Path(path_to_files).glob('file*dat'))
hit_df, status_df = helper.process_tamalero_outputs(files)

In [ ]:
hit_df.info()

In [ ]:
status_df.info()

In [ ]:
with open("./board_configs_yaml/CERN_TB_H6_2026April_CELip.yaml", "r") as file:

    fig_configs = yaml.safe_load(file)

print(fig_configs.keys())

given_run = 'run1'
selected_fig_config = fig_configs[given_run]

for id in selected_fig_config.keys():
    selected_fig_config[id]['title'] = f"{selected_fig_config[id]['name']} HV{selected_fig_config[id]['HV']}V OS:{selected_fig_config[id]['offset']}"

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['angle']}deg'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' temp:{selected_fig_config[id]['temp']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' IRRAD:{selected_fig_config[id]['irrad']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['note']}'
    except:
        pass

board_names = []
for key, val in selected_fig_config.items():
    print(key, val)
    board_names.append(val['name'])

In [ ]:
helper.plot_occupany_map(hit_df, board_info=selected_fig_config, tb_loc='desy')

In [ ]:
h_inclusive = helper.return_hist(input_df=hit_df, board_info=selected_fig_config, hist_bins=[100, 128, 128])
chip_figtitles = [val['title'] for _, val in selected_fig_config.items()]
helper.plot_1d_TDC_histograms(input_hist=h_inclusive, tb_loc='desy', fig_tag=chip_figtitles, slide_friendly=True)

## Status Dashboard

In [ ]:
import matplotlib.pyplot as plt
import mplhep as mpl
mpl.style.use('CMS')
import matplotlib.colors as mcolors

In [ ]:
# 1. Setup Status Mapping (Universal)
val_map = {0: 0, 16: 1, 32: 2, 48: 3}
status_colors = ['#4caf50', '#ffcc99', '#ff9800', '#e53935']
status_cmap = mcolors.ListedColormap(status_colors)
status_labels = ['L1 buffer < 64', '64-95', '96-127', 'Full']

hit_colors = ['#1f78b4', '#a65628', '#7570b3', '#e7298a']

global_max_events = status_df.groupby('board').size().max() * 1.01
global_max_hits = status_df['hits'].max()
left_space = global_max_events * 0.01 * -1.

# 2. Loop through each board and create a unique figure
for i, name in enumerate(board_names):
    tmp_df = status_df[status_df['board'] == i].reset_index(drop=True)
    if tmp_df.empty:
        continue

    # Create a figure for this specific board
    # height_ratios [1, 5] makes the ribbon 1/5th the height of the hit plot
    fig, (ax_ribbon, ax_hits) = plt.subplots(2, 1, figsize=(15, 10),
                                             sharex=True,
                                             gridspec_kw={'height_ratios': [1, 4]})

    # --- Top Plot: Status Ribbon ---
    status_row = tmp_df['status'].map(val_map).values.reshape(1, -1)
    ax_ribbon.imshow(status_row, cmap=status_cmap, aspect='auto',
                     interpolation='nearest', vmin=0, vmax=3)

    # Styling Ribbon
    ax_ribbon.set_yticks([])
    ax_ribbon.set_title(f'Status Dashboard: {name}', fontsize=18, fontweight='bold', pad=10, loc='left')
    ax_ribbon.set_ylabel('Status', labelpad=15, loc='center', fontsize=19)

    # --- Bottom Plot: Hits Line ---
    ax_hits.plot(tmp_df.index, tmp_df['hits'], marker='.', color=hit_colors[i], linewidth=0.8, alpha=0.9)

    # Styling Hits
    ax_hits.set_ylabel('Number of Hits', loc='center')
    ax_hits.set_xlabel('Event')
    ax_hits.grid(True, which='both', linestyle='--', alpha=0.4)

    ax_hits.set_xlim(left_space, global_max_events)
    ax_hits.set_ylim(-0.5, 32)

    # Add a local legend for status bits to each figure
    patches = [plt.Rectangle((0,0),1,1, color=status_colors[j]) for j in range(4)]
    ax_ribbon.legend(patches, status_labels, loc='upper right',
                     bbox_to_anchor=(1.0, 1.25), ncol=4, frameon=False, fontsize=13)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.01) # Connect the two plots closely

## Print number of events

In [ ]:
from itertools import combinations

# 1. Get total unique events (nunique() is faster than len(df[...].unique()))
tot_events = hit_df['evt'].nunique()

# 2. Extract sets of unique events for each board
# Mapping: Board 0 -> Ch 1, Board 1 -> Ch 2, etc.
# Python Sets are significantly faster for intersections than numpy arrays
events_per_ch = {
    i + 1: set(hit_df.loc[hit_df['board'] == i, 'evt'])
    for i in range(4)
}

print(f"Total Events: {tot_events}\n")

# 3. Dynamically calculate intersections for groups of 1, 2, 3, and 4
for r in range(1, 5):
    for combo in combinations(events_per_ch.keys(), r):
        # Intersect the sets for the current combination
        common_events = set.intersection(*(events_per_ch[c] for c in combo))
        count = len(common_events)

        # Calculate percentage (with safe division check)
        pct = (count / tot_events * 100) if tot_events else 0

        # Format the channel names dynamically (e.g., "Ch1+Ch2")
        ch_names = "+".join([f"Ch{c}" for c in combo])

        # Note: Changed your :02f to :.2f for standard 2-decimal-place formatting
        print(f"{ch_names} Events: {count} ({pct:.2f}%)")

    print("") # Print a blank line after each group size finishes

In [ ]:
# import numpy
# ch1_evts = df.loc[df['board'] == 0]['evt'].unique()
# ch2_evts = df.loc[df['board'] == 1]['evt'].unique()
# ch3_evts = df.loc[df['board'] == 2]['evt'].unique()
# ch4_evts = df.loc[df['board'] == 3]['evt'].unique()

# ch1ch2_evts = numpy.intersect1d(ch1_evts, ch2_evts)
# ch1ch3_evts = numpy.intersect1d(ch1_evts, ch3_evts)
# ch1ch4_evts = numpy.intersect1d(ch1_evts, ch4_evts)
# ch2ch3_evts = numpy.intersect1d(ch2_evts, ch3_evts)
# ch2ch4_evts = numpy.intersect1d(ch2_evts, ch4_evts)
# ch3ch4_evts = numpy.intersect1d(ch3_evts, ch4_evts)

# ch1ch2ch3_evts = numpy.intersect1d(ch1ch2_evts, ch3_evts)
# ch1ch2ch4_evts = numpy.intersect1d(ch1ch2_evts, ch4_evts)
# ch1ch3ch4_evts = numpy.intersect1d(ch1ch3_evts, ch4_evts)
# ch2ch3ch4_evts = numpy.intersect1d(ch2ch3_evts, ch4_evts)

# ch1ch2ch3ch4_evts = numpy.intersect1d(ch1ch2ch3_evts, ch4_evts)

In [ ]:
# tot_events = len(df['evt'].unique())

# ch1_df = df.loc[df['evt'].isin(ch1_evts)]
# ch2_df = df.loc[df['evt'].isin(ch2_evts)]
# ch3_df = df.loc[df['evt'].isin(ch3_evts)]
# ch4_df = df.loc[df['evt'].isin(ch4_evts)]
# events_ch1 = len(ch1_df['evt'].unique())
# events_ch2 = len(ch2_df['evt'].unique())
# events_ch3 = len(ch3_df['evt'].unique())
# events_ch4 = len(ch4_df['evt'].unique())

# ch1ch2_df = df.loc[df['evt'].isin(ch1ch2_evts)]
# ch1ch3_df = df.loc[df['evt'].isin(ch1ch3_evts)]
# ch1ch4_df = df.loc[df['evt'].isin(ch1ch4_evts)]
# ch2ch3_df = df.loc[df['evt'].isin(ch2ch3_evts)]
# ch2ch4_df = df.loc[df['evt'].isin(ch2ch4_evts)]
# ch3ch4_df = df.loc[df['evt'].isin(ch3ch4_evts)]
# events_ch1ch2 = len(ch1ch2_df['evt'].unique())
# events_ch1ch3 = len(ch1ch3_df['evt'].unique())
# events_ch1ch4 = len(ch1ch4_df['evt'].unique())
# events_ch2ch3 = len(ch2ch3_df['evt'].unique())
# events_ch2ch4 = len(ch2ch4_df['evt'].unique())
# events_ch3ch4 = len(ch3ch4_df['evt'].unique())

# ch1ch2ch3_df = df.loc[df['evt'].isin(ch1ch2ch3_evts)]
# ch1ch2ch4_df = df.loc[df['evt'].isin(ch1ch2ch4_evts)]
# ch1ch3ch4_df = df.loc[df['evt'].isin(ch1ch3ch4_evts)]
# ch2ch3ch4_df = df.loc[df['evt'].isin(ch2ch3ch4_evts)]
# events_ch1ch2ch3 = len(ch1ch2ch3_df['evt'].unique())
# events_ch1ch2ch4 = len(ch1ch2ch4_df['evt'].unique())
# events_ch1ch3ch4 = len(ch1ch3ch4_df['evt'].unique())
# events_ch2ch3ch4 = len(ch2ch3ch4_df['evt'].unique())

# ch1ch2ch3ch4_df = df.loc[df['evt'].isin(ch1ch2ch3ch4_evts)]
# events_ch1ch2ch3ch4 = len(ch1ch2ch3ch4_df['evt'].unique())


# print(f"Total Events: {tot_events}")
# print("")
# print(f"Ch1 Events: {events_ch1} ({events_ch1/float(tot_events)*100:02f})")
# print(f"Ch2 Events: {events_ch2} ({events_ch2/float(tot_events)*100:02f})")
# print(f"Ch3 Events: {events_ch3} ({events_ch3/float(tot_events)*100:02f})")
# print(f"Ch4 Events: {events_ch4} ({events_ch4/float(tot_events)*100:02f})")
# print("")
# print(f"Ch1+Ch2 Events: {events_ch1ch2} ({events_ch1ch2/float(tot_events)*100:02f})")
# print(f"Ch1+Ch3 Events: {events_ch1ch3} ({events_ch1ch3/float(tot_events)*100:02f})")
# print(f"Ch1+Ch4 Events: {events_ch1ch4} ({events_ch1ch4/float(tot_events)*100:02f})")
# print(f"Ch2+Ch3 Events: {events_ch2ch3} ({events_ch2ch3/float(tot_events)*100:02f})")
# print(f"Ch2+Ch4 Events: {events_ch2ch4} ({events_ch2ch4/float(tot_events)*100:02f})")
# print(f"Ch3+Ch4 Events: {events_ch3ch4} ({events_ch3ch4/float(tot_events)*100:02f})")
# print("")
# print(f"Ch1+Ch2+Ch3 Events: {events_ch1ch2ch3} ({events_ch1ch2ch3/float(tot_events)*100:02f})")
# print(f"Ch1+Ch2+Ch4 Events: {events_ch1ch2ch4} ({events_ch1ch2ch4/float(tot_events)*100:02f})")
# print(f"Ch1+Ch3+Ch4 Events: {events_ch1ch3ch4} ({events_ch1ch3ch4/float(tot_events)*100:02f})")
# print(f"Ch2+Ch3+Ch4 Events: {events_ch2ch3ch4} ({events_ch2ch3ch4/float(tot_events)*100:02f})")
# print("")
# print(f"Ch1+Ch2+Ch3+Ch4 Events: {events_ch1ch2ch3ch4} ({events_ch1ch2ch3ch4/float(tot_events)*100:02f})")